In [2]:
import functools
import math
from pprint import pprint
from collections.abc import Callable

In [3]:
def unrank(n, k, r, last=0):
  if k == 0:
    return []
  x = math.comb(n-1, k-1)
  if r < x:
    return [last+1] + unrank(n-1, k-1, r, last+1)
  else:
    return unrank(n-1, k, r - x, last+1)

In [4]:

@functools.lru_cache(maxsize=None)
# number of ways to partition n elements into k non-empty groups
def nombre_de_stirling(n, k):
  # There is exactly 1 way to partition 0 elements into 0 groups.
  if n == 0 and k == 0: return 1
  # ex: Partition 0 elements into 3 groups ->  impossible
  if n == 0 or k == 0: return 0
  return nombre_de_stirling(n-1, k-1) + k*nombre_de_stirling(n-1, k)


In [5]:
print(nombre_de_stirling(3,2))
print(nombre_de_stirling(50,25))

3
7453802153273200083379626234837625465912500


In [6]:
# distinct elements into k identical boxes,
# where order matters inside each box (each box is a list),
# and no box is empty.
@functools.lru_cache(maxsize=None)
def nombre_de_lah(n,k):
    if n == 0 and k == 0: return 1 
    if n == 0 or k == 0: return 0
    if n == k: return 1 
    return nombre_de_lah(n-1,k-1) + (n+k-1)*nombre_de_lah(n-1,k)

nombre_de_lah(50, 25)


123931787241057448522969459838565791518810963968000000

In [19]:

for i in range(9) :
    for j in range(i+1) :
        print(nombre_de_lah(i,j),end=" ")
    print("")

1 
0 1 
0 2 1 
0 6 6 1 
0 24 36 12 1 
0 120 240 120 20 1 
0 720 1800 1200 300 30 1 
0 5040 15120 12600 4200 630 42 1 
0 40320 141120 141120 58800 11760 1176 56 1 


In [20]:
def unranking_stirling(n,k,r) : 
  if n == 0 or k == 0:
    return []
  if r < nombre_de_stirling(n-1,k-1): 
    return unranking_stirling(n-1,k-1,r) + [[n]]
  else:
    r = r - nombre_de_stirling(n-1,k-1)
    pos, r = divmod(r,nombre_de_stirling(n-1,k))
    us = unranking_stirling(n-1,k,r)
    us[pos].append(n)
    return us

print(unranking_stirling(5, 3, 15))
     
# TODO test verifier que les combs sont diff+bien forme+ bon nb


[[1], [2, 3, 5], [4]]


In [21]:
def unranking_lah(n, k, r):
    if n == 0 and k == 0:
        return []
    if n == 0 or k == 0:
        return [] # None?
    if n == k: # bloc for each number
        return [[x+1] for x in range(n)]
    
    fst_case_cnt = nombre_de_lah(n-1, k-1)
    if r < fst_case_cnt: # Singleton
        return unranking_lah(n-1, k-1, r) + [[n]]
    else: # Inserer dans une des boites
        r2 = r - fst_case_cnt
        pos, r = divmod(r2, nombre_de_lah(n-1, k))
        res = unranking_lah(n-1, k, r)
        # trouver l'indice
        for boite in res:
            if pos < len(boite) + 1:
                boite.insert(pos, n)
                break
            pos -= len(boite) + 1
        return res

In [22]:
n, k = 4, 3
x = [unranking_lah(n, k, i) for i in range(nombre_de_lah(n, k))]
print(f"Length: {len(x)}")
pprint(x)

Length: 12
[[[2, 1], [3], [4]],
 [[1, 2], [3], [4]],
 [[3, 1], [2], [4]],
 [[1, 3], [2], [4]],
 [[1], [3, 2], [4]],
 [[1], [2, 3], [4]],
 [[4, 1], [2], [3]],
 [[1, 4], [2], [3]],
 [[1], [4, 2], [3]],
 [[1], [2, 4], [3]],
 [[1], [2], [4, 3]],
 [[1], [2], [3, 4]]]


In [43]:
def unrank_perm(n, r): #unrank de n! different orders
    elems = list(range(1, n+1))
    #print(elems)
    perm = []
    for m in range(n, 0, -1):
        f = math.factorial(m-1)
        idx, r = divmod(r, f)
        perm.append(elems.pop(idx))
    return perm

def unranking_ordered_stirling(n, k, r):
    # Base cases
    if k == 1:
        return [list(range(1, n+1))]          # one block (order inside doesn't matter)
    if k == n:
        perm = unrank_perm(n, r)              # order of singleton blocks
        return [[x] for x in perm]

    # Case A: n is a singleton block [n], inserted in one of k positions
    cntA = k * ordered_stirling_numbers(n-1, k-1)
    if r < cntA:
        pos, r = divmod(r, ordered_stirling_numbers(n-1, k-1))
        res = unranking_ordered_stirling(n-1, k-1, r)
        res.insert(pos, [n])
        return res

    # Case B: n joins one of the k existing blocks
    r = r - cntA
    pos, r = divmod(r, ordered_stirling_numbers(n-1, k))
    res = unranking_ordered_stirling(n-1, k, r)
    res[pos].append(n)
    # res[pos].sort()                              
    return res

# quick sanity check
n, k = 5, 3
all_os = [unranking_ordered_stirling(n, k, r) for r in range(ordered_stirling_numbers(n, k))]
print(len(all_os), ordered_stirling_numbers(n, k))


150 150


In [24]:
def forme_canonique(x, ordered=True):
    res = [sorted(boite) if not ordered
           else boite.copy()
           for boite in x]
    res.sort(key=min)
    return res

In [25]:
x = [[6, 3], [5, 1, 2], [4]]
print(x)
print(f"Not ordered:{forme_canonique(x, ordered=False)}")
print(f"Ordered:{forme_canonique(x, ordered=True)}")

[[6, 3], [5, 1, 2], [4]]
Not ordered:[[1, 2, 5], [3, 6], [4]]
Ordered:[[5, 1, 2], [6, 3], [4]]


In [26]:
@functools.lru_cache
def ordered_stirling_numbers(n, k):
    if k == 1 or k == n:
        return math.factorial(k)
    return k*(ordered_stirling_numbers(n-1, k-1) \
            + ordered_stirling_numbers(n-1, k))

In [27]:
ordered_stirling_numbers(8, 3)

5796

In [ ]:
@functools.lru_cache
def ordered_lah_numbers(n, k):
    if k == 1 or k == n:
        return math.factorial(n)
    return k*ordered_lah_numbers(n-1, k-1) \
        + (n-1 + k) * ordered_lah_numbers(n-1, k)

In [ ]:
ordered_lah_numbers(8, 6)

846720

In [32]:
@functools.lru_cache
def int_partitions(n, k):
    if n < 1:
        return 0
    if k == 1 or n == k:
        return 1
    return int_partitions(n-1, k-1) + int_partitions(n-k, k)

In [ ]:
print(int_partitions(9, 3))
print(int_partitions(90, 21))

7
2801502


In [31]:
def unrank_int_partitions(n, k, r) -> list[int]:
    if n < 1:
        return []
    if k == 1:
        return [n]
    if n == k:
        return [1]*n
    if r < int_partitions(n-1, k-1):
        return [1] + unrank_int_partitions(n-1, k-1, r)
    else:
        r2 = r - int_partitions(n-1, k-1)
        res = unrank_int_partitions(n-k, k, r2)
        res = [i+1 for i in res]
        return res

In [30]:
def check_property_of_partitions(
    description: str, ps, prop_lambda: Callable):
    '''
    `description`: printed before the test
    `ps`: list of partitions
    `prop_lambda`: function that takes a partition,
        and returns a boolean value
    '''
    print(description)
    for p in ps:
        if prop_lambda(p):
            print("-- No, exiting...")
            break
    print("-- Yes.")

def test_unrank_partitions(n, k):
    x = int_partitions(n, k)
    res = [unrank_int_partitions(n, k, i) \
            for i in range(x)]
    print(f"n: {n}, k: {k}, length of unrank: {len(res)}")
    check_property_of_partitions(\
        "Each partition has K blocks?",
        res, lambda p: len(p) != k
    )
    check_property_of_partitions(\
        "Sum of each partition = n?",
        res, lambda p: sum(p) != n
    )
    print()
    pprint(sorted(res), width=40)

In [33]:
n, k = 11, 4
test_unrank_partitions(n, k)

n: 11, k: 4, length of unrank: 11
Each partition has K blocks?
-- Yes.
Sum of each partition = n?
-- Yes.

[[1, 1, 1, 8],
 [1, 1, 2, 7],
 [1, 1, 3, 6],
 [1, 1, 4, 5],
 [1, 2, 2, 6],
 [1, 2, 3, 5],
 [1, 2, 4, 4],
 [1, 3, 3, 4],
 [2, 2, 2, 5],
 [2, 2, 3, 4],
 [2, 3, 3, 3]]


In [58]:
# Def 28
def less_than_lex(A, B):
    for i in range(len(A)):
        if (i >= len(B)):
            return True
        if A[i] < B[i]:
            return True
        elif A[i] > B[i]:
            return False
        # if ==, continue
    return True
        

In [ ]:
less_than_lex([[1, 4], [2, 5]], [[1, 4],[2, 5],[3]])

False